In [1]:
import logging
import os
import re
from types import SimpleNamespace
import ruamel.yaml

import numpy as np
import pandas as pd
import geopandas as gpd
import pypsa
import pytz
import ruamel.yaml
import xarray as xr
from helpers import (
    create_dummy_data,
    create_network_topology,
    cycling_shift,
    locate_bus,
    mock_snakemake,
    override_component_attrs,
    prepare_costs,
    safe_divide,
    three_2_two_digits_country,
    two_2_three_digits_country,
    lossy_bidirectional_links,
    three_2_two_digits_country,
)
from helpers_offgrid import(
    add_nice_carrier_names,
    calculate_annuity,
    _add_missing_carriers_from_costs,
    load_costs,
    create_import_profile,
    add_esc_shipping,
    add_shipping_meoh,
    add_shipping_lnh3,
    add_shipping_lh2,
    create_esc_network,
    )
from prepare_transport_data import prepare_transport_data
from add_export_supply_chain import (
    get_efficiency, 
    read_efficiencies,
    select_ports,
    get_shipping_distance,
    parse_json_to_geodataframe,
    )
# from add_export_supply_chain import *


/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/networkclustering.py:16: UserWarning: The namespace `pypsa.networkclustering` is deprecated and will be removed in PyPSA v0.24. Please use `pypsa.clustering.spatial instead`. 
  warnings.warn(


In [2]:
# # Old Costs +  Monthly matching + AP
# n0_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_at_port_20241104h2Port/postnetworks/elec_s_456_ec_lv1.1_Co2L_3H_2050_0.091_AP_0export_shipping_lnh3.nc'
# costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050_previously_used.csv'
# file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config_2050_real.yaml'
# file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth_realistic_2050.yaml'

# # Old Costs +  Monthly matching + NZ
# n0_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_at_port_20241104h2Port/postnetworks/elec_s_590_ec_lv1.25_Co2L_3H_2050_0.045_NZ_0export_shipping_lnh3.nc'
# costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050_previously_used.csv'
# file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config_2050_opt.yaml'
# file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth_optimistic_2050.yaml'



#---------------------


# # New Costs +  Hourly matching + AP
# n0_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_at_port_20241101h2Port/postnetworks/elec_s_444_ec_lv1.1_Co2L_3H_2050_0.091_AP_0export_shipping_lnh3.nc'
# costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050_new_from_onlilne_version.csv'
# file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config_2050_real.yaml'
# file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth_realistic_2050.yaml'


# # New Costs +  Hourly matching + NZ
# n0_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_at_port_20241101h2Port/postnetworks/elec_s_578_ec_lv1.25_Co2L_3H_2050_0.045_NZ_0export_shipping_lnh3.nc'
# costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050_new_from_onlilne_version.csv'
# file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config_2050_opt.yaml'
# file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth_optimistic_2050.yaml'


#---------------------

# # New Costs +  Hourly matching + AP + DEA electrolizer price
# n0_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_free_20241105h2Free/postnetworks/elec_s_474_ec_lv1.1_Co2L_3H_2050_0.091_AP_0export_shipping_lnh3.nc'
# costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050_new_DEA_electrolizer_price.csv'
# file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config_2050_real.yaml'
# file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth_realistic_2050.yaml'





#---------------------

# New Costs +  Hourly matching + AP + LFS electrolizer price
n0_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_at_port_20241113h2Port/postnetworks/elec_s_480_ec_lv1.1_Co2L_3H_2050_0.091_AP_0export_shipping_lnh3.nc'
costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050_new_LFS_electrolizer_price.csv'
file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config_2050_real.yaml'
file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth_realistic_2050.yaml'




In [3]:
# costs_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/costs_2050.csv'

In [4]:
h2export_all_quantities: [0, 10, 20, 50, 80, 100, 150]
h2export = 100 #TWh
offgrid_scenario = "RESprofileatport" # "RESprofileatport" or "RESprofilecombined"

### Load config 

In [5]:
# file_path = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.yaml'

# Read the YAML file
yaml = ruamel.yaml.YAML()


with open(file_path, 'r') as file:
    yaml_content = yaml.load(file)

    techs = yaml_content["custom_data"]["renewables"]
    year = yaml_content["scenario"]["planning_horizons"][0]
    dr = yaml_content["costs"]["discountrate"][0]
    sopts = yaml_content["scenario"]["sopts"]
    demand_sc = yaml_content["scenario"]["demand"][0]
    country = yaml_content["countries"]

    yaml_content["export"]["esc_scenarios"]["synthesis"] = 'at_port'


# file_path_pypsa_earth = '/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/config.pypsa-earth.yaml'

with open(file_path_pypsa_earth, 'r') as file2:
    yaml_content2 = yaml.load(file2)



### Get the centroid of Turkey for buses location

In [6]:
export_ports_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/export_ports.csv"
ports = pd.read_csv(
    export_ports_path,
    index_col=None,
    keep_default_na=False,
).squeeze()

ports = ports[ports.country.isin(country)]

exp_ports = ports.copy()

port_buses_n0 = {'Samsun':'TR.63_1_AC', 'Mersin':'TR.58_1_AC', 'Aliaga':'TR.41_1_AC'}

exp_ports.set_index("name", inplace=True)
exp_ports["bus"] = exp_ports.index.map(port_buses_n0)

# Reset the index but keep 'name' as a column, and set 'bus' as the new index
exp_ports.reset_index(inplace=True)
exp_ports.set_index("bus", inplace=True)




In [7]:
exp_ports

,name,country,fraction,y,x
bus,,,,,
TR.63_1_AC,Samsun,TR,0.333,41.29961,36.34982
TR.58_1_AC,Mersin,TR,0.333,36.79983,34.63333
TR.41_1_AC,Aliaga,TR,0.333,38.83316,26.93334


In [8]:
n0 = pypsa.Network(n0_path)


/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value

In [9]:
# nodes_n0 = n0.buses.filter(regex='_AC$', axis=0)
# nodes_n0

### Create elec network and assign RES generators

In [10]:
# Create an empty n
overrides = override_component_attrs("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/override_component_attrs")
n = pypsa.Network(override_component_attrs=overrides)
n.name = "PyPSA-Earth"
n.set_snapshots(pd.date_range(freq=sopts[0], **yaml_content["snapshots"]))
n.snapshot_weightings[:] *= 8760.0 / n.snapshot_weightings.sum()
n.meta = yaml_content

Nyears = n.snapshot_weightings.objective.sum() / 8760.0

#--------------------- options
# OPTION 1: Get all AC nodes in n0
# nodes_n0 = n0.buses.filter(regex='_AC$', axis=0)

# OPTION 2: Get port AC nodes in n0
# nodes_n0 = n0.buses.filter(regex='_AC$', axis=0).filter(exp_ports.index, axis=0)

# OPTION 3: Get top 3 CF and ports AC 
mean_solar_CF = n0.generators_t.p_max_pu.filter(like='_AC solar').mean()
mean_solar_CF.index = mean_solar_CF.index.str.split(' ').str[0]
mean_solar_CF = mean_solar_CF.loc[mean_solar_CF.index.difference(exp_ports.index)] # Perform the difference operation to exclude indexes in exp_ports
nodes_n0 = mean_solar_CF.sort_values(ascending=False).head(3).index
nodes_n0 = nodes_n0.union(exp_ports.index)
nodes_n0 = n0.buses.filter(regex='_AC$', axis=0).filter(nodes_n0, axis=0)
#---------------------


n.add("Carrier", "H2")
# n.add("Carrier", "H2")

# Add all electricty buses from n0
n.madd(
    "Bus",
    nodes_n0.index,
    carrier="AC",
    v_nom=nodes_n0.v_nom.values,
    x=nodes_n0.x.values,
    y=nodes_n0.y.values,
    location=nodes_n0.index,
    country='TR',
)

# # Add 3 electricity bus locations to attach ESC to them
# n.madd(
#     "Bus",
#     exp_ports.index,
#     carrier="AC",
#     v_nom=380,
#     x=exp_ports.x.values,
#     y=exp_ports.y.values,
#     location=exp_ports.index,
#     country='TR',
# )
  
nodes = n.buses[n.buses.carrier == "AC"].index



# Load reference network to get RES data from:
n0 = pypsa.Network(n0_path)

techs= ['solar', 'onwind', 'onwind2']




# Attach RES data into n
for tech in techs:
    custom_res = n0.generators[(n0.generators.carrier == tech) & (n0.generators.bus.isin(nodes))]
    custom_res_index = custom_res.index
    custom_res_t = n0.generators_t.p_max_pu.filter(custom_res_index)
    custom_res_t.columns = custom_res_t.columns.str.replace(' '+tech, '', regex=False)
    # profile= pd.DataFrame(custom_res_t.mean(axis=1), columns=["TR_AC"])
    profile= custom_res_t.copy()
    print(profile.sum())


    n.madd(
        "Generator",
        nodes,
        " " + tech,
        bus=nodes,
        carrier=tech,
        p_nom_extendable=True,
        p_nom_max=(custom_res["p_nom_max"] - custom_res["p_nom_opt"]).values,
        # p_nom_max=custom_res["p_nom_max"].values,
        capital_cost=custom_res["capital_cost"].values,
        efficiency=1.0,
        p_max_pu=profile,
        lifetime=custom_res["lifetime"].iloc[0],
    )

/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
/nfs/home/edd32710/.conda/envs/PES_Model/lib/python3.10/site-packages/pypsa/components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value

Generator
TR.41_1_AC    479.534223
TR.44_1_AC    501.920927
TR.58_1_AC    496.792805
TR.63_1_AC    393.689282
TR.68_1_AC    509.901249
TR.8_1_AC     501.110389
dtype: float64
Generator
TR.41_1_AC    452.354609
TR.44_1_AC    245.969501
TR.58_1_AC    149.326372
TR.63_1_AC    143.873744
TR.68_1_AC    430.755260
TR.8_1_AC      85.526039
dtype: float64
Generator
TR.41_1_AC    32.832820
TR.44_1_AC    27.127488
TR.58_1_AC    13.246298
TR.63_1_AC    31.099900
TR.68_1_AC    56.002722
TR.8_1_AC      5.738726
dtype: float64


In [11]:
# Prepare the costs dataframe
costs = prepare_costs(
    costs_path,
    yaml_content["costs"]["USD2013_to_EUR2013"],
    dr,
    Nyears,
    yaml_content["costs"]["lifetime"],
)

In [12]:
costs2 = load_costs(
        costs_path,
        yaml_content2["costs"],
        yaml_content2["electricity"],
        Nyears,
    )


In [13]:
elec_opts = yaml_content2["electricity"]
carriers = pd.Index(elec_opts["extendable_carriers"]["Generator"])
_add_missing_carriers_from_costs(n, costs, carriers)
add_nice_carrier_names(n, config=yaml_content2)

### Add Hydrogen

In [14]:
# Add Hydrogen bus
n.madd(
    "Bus",
    nodes + " H2",
    carrier="H2",
    x=nodes_n0.x.values,
    y=nodes_n0.y.values,
    location=nodes,
)


n.madd(
    "Link",
    nodes + " H2 Electrolysis",
    bus1=nodes + " H2",
    bus0=nodes,
    p_nom_extendable=True,
    carrier="H2 Electrolysis",
    efficiency=costs.at["electrolysis", "efficiency"],
    capital_cost=costs.at["electrolysis", "fixed"],
    lifetime=costs.at["electrolysis", "lifetime"],
)

n.madd(
    "Link",
    nodes + " H2 Fuel Cell",
    bus0=nodes + " H2",
    bus1=nodes,
    p_nom_extendable=True,
    carrier="H2 Fuel Cell",
    efficiency=costs.at["fuel cell", "efficiency"],
    # NB: fixed cost is per MWel
    capital_cost=costs.at["fuel cell", "fixed"]
    * costs.at["fuel cell", "efficiency"],
    lifetime=costs.at["fuel cell", "lifetime"],
)


Index(['TR.41_1_AC H2 Fuel Cell', 'TR.44_1_AC H2 Fuel Cell',
       'TR.58_1_AC H2 Fuel Cell', 'TR.63_1_AC H2 Fuel Cell',
       'TR.68_1_AC H2 Fuel Cell', 'TR.8_1_AC H2 Fuel Cell'],
      dtype='object', name='Bus')

,carrier,v_nom,x,y,location,country,control,sub_network,type,unit,v_mag_pu_max,v_mag_pu_min,v_mag_pu_set
Bus,,,,,,,,,,,,,
TR.41_1_AC,AC,380.0,27.192487,38.448383,TR.41_1_AC,TR,PQ,,,MWh,inf,0.0,1.0
TR.44_1_AC,AC,380.0,33.164275,37.078695,TR.44_1_AC,TR,PQ,,,MWh,inf,0.0,1.0
TR.58_1_AC,AC,380.0,33.942093,36.608450,TR.58_1_AC,TR,PQ,,,MWh,inf,0.0,1.0
TR.63_1_AC,AC,380.0,36.192950,41.213584,TR.63_1_AC,TR,PQ,,,MWh,inf,0.0,1.0
TR.68_1_AC,AC,380.0,38.971293,37.118002,TR.68_1_AC,TR,PQ,,,MWh,inf,0.0,1.0
TR.8_1_AC,AC,380.0,30.980096,36.803845,TR.8_1_AC,TR,PQ,,,MWh,inf,0.0,1.0
TR.41_1_AC H2,H2,1.0,27.192487,38.448383,TR.41_1_AC,,PQ,,,MWh,inf,0.0,1.0
TR.44_1_AC H2,H2,1.0,33.164275,37.078695,TR.44_1_AC,,PQ,,,MWh,inf,0.0,1.0
TR.58_1_AC H2,H2,1.0,33.942093,36.608450,TR.58_1_AC,,PQ,,,MWh,inf,0.0,1.0


In [16]:
# Efficiencies
efficiencies_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/esc_data/efficiencies.csv"
efficiencies = read_efficiencies(
    efficiencies_path, yaml_content["scenario"]["planning_horizons"]
)



h2_pot = n0.stores[(n0.stores.carrier == "H2 UHS") & (n0.stores.bus.str.contains('|'.join(nodes)))].e_nom_max
h2_capital_cost = n0.stores[(n0.stores.carrier == "H2 UHS") & (n0.stores.bus.str.contains('|'.join(nodes)))].capital_cost


n.madd(
    "Bus",
    nodes + " H2 UHS",
    location=nodes,
    carrier="H2 UHS",
    x=nodes_n0.x.values,
    y=nodes_n0.y.values,
)

n.madd(
    "Store",
    nodes + " H2 UHS",
    bus=nodes + " H2 UHS",
    e_nom_extendable=True,
    e_nom_max=h2_pot,
    e_cyclic=True,
    carrier="H2 UHS",
    capital_cost=h2_capital_cost,
    lifetime=costs.at["hydrogen storage underground", "lifetime"],
)

n.madd(
    "Link",
    nodes + " H2 UHS charger",
    bus0=nodes + " H2",
    bus1=nodes + " H2 UHS",
    carrier="H2 UHS charger",
    # efficiency=costs.at["battery inverter", "efficiency"] ** 0.5,
    # capital_cost=costs.at["battery inverter", "fixed"],
    p_nom_extendable=True,
    # lifetime=costs.at["battery inverter", "lifetime"],
)

n.madd(
    "Link",
    nodes + " H2 UHS discharger",
    bus0=nodes + " H2 UHS",
    bus1=nodes + " H2",
    carrier="H2 UHS discharger",
    efficiency=1,
    # capital_cost=costs.at["battery inverter", "fixed"],
    p_nom_extendable=True,
    # lifetime=costs.at["battery inverter", "lifetime"],
)



# Gasious hydrogen storage overground (on all nodes)
n.madd(
    "Bus", 
    nodes, 
    suffix=" H2 gas storage", 
    carrier="H2 gas storage", 
    location=nodes, 
    unit="MWh", 
    country=n.buses.loc[list(nodes)].country.values,
    x=n.buses.loc[list(nodes)].x.values,
    y=n.buses.loc[list(nodes)].y.values,
)

n.madd(
    "Link",
    nodes + " H2 storage compressor (exp)",
    bus0=nodes + " H2",
    bus1=nodes + " H2 gas storage",
    bus2=nodes,
    carrier="H2 storage compressor",
    efficiency=get_efficiency(efficiencies, 'H2 storage compressor', 'H2 (g)', 'H2 (g) storage'),
    efficiency2= (
        (-1)
        * get_efficiency(efficiencies, 'H2 storage compressor', 'H2 (g)', 'H2 (g) storage')
        / get_efficiency(efficiencies, 'H2 storage compressor', 'electricity', 'H2 (g) storage')
    ),
    capital_cost=costs.at["H2 (g) fill compressor station", "fixed"], # Needs to be checked . This is only for pipelines compressors.
    p_nom_extendable=True,
    lifetime=costs.at["H2 (g) fill compressor station", "lifetime"],
    p_min_pu=0,
    p_max_pu=1,
)

n.madd(
    "Link",
    nodes + " H2 storage unstoring (exp)",
    bus0=nodes + " H2 gas storage",
    bus1=nodes + " H2",
    # bus2=spatial.nodes,
    carrier="H2 storage unstoring",
    efficiency=get_efficiency(efficiencies, 'H2 storage unstoring', 'H2 (g) storage', 'H2 (g)'),
    # efficiency2= (-1),
    p_nom_extendable=True,
    p_min_pu=0,
    p_max_pu=1,
)

n.madd(
    "Store",
    nodes + " H2 Gas Store Tank",
    bus=nodes + " H2 gas storage",
    e_nom_extendable=True,
    e_cyclic=True,
    carrier="H2 Gas Store Tank",
    marginal_cost=0,
    capital_cost=costs.at[
        "hydrogen storage tank type 1 including compressor", "fixed"
    ],
    lifetime=costs.at["hydrogen storage tank type 1 including compressor", "lifetime"],
)


# get hydrogen export buses/ports
hydrogen_buses_ports = n.buses[(n.buses.carrier == "H2") & (n.buses.location.isin(exp_ports.index))]

# List of ports electrictiy buses
exp_nodes = hydrogen_buses_ports.location.values

### Attach Geothermal

In [17]:
# geothermal_pot = n0.generators[n0.generators.carrier == "geothermal"].p_nom_max.sum()

# n.madd(
#     "Generator",
#     nodes,
#     " " + "geothermal",
#     bus=nodes,
#     carrier="geothermal",
#     p_nom_extendable=True,
#     p_nom_max=geothermal_pot,
#     capital_cost=costs2.at["geothermal", "capital_cost"],
#     marginal_cost=costs2.at["geothermal", "marginal_cost"],
#     efficiency=costs2.at["geothermal", "efficiency"],
# )

### Attach Battery storage

In [18]:
n.add("Carrier", "battery")


n.madd(
    "Bus",
    nodes + " battery",
    location=nodes,
    carrier="battery",
    x=nodes_n0.x.values,
    y=nodes_n0.y.values,
)

n.madd(
    "Store",
    nodes + " battery",
    bus=nodes + " battery",
    e_cyclic=True,
    e_nom_extendable=True,
    carrier="battery",
    capital_cost=costs.at["battery storage", "fixed"],
    lifetime=costs.at["battery storage", "lifetime"],
)

n.madd(
    "Link",
    nodes + " battery charger",
    bus0=nodes,
    bus1=nodes + " battery",
    carrier="battery charger",
    efficiency=costs.at["battery inverter", "efficiency"] ** 0.5,
    capital_cost=costs.at["battery inverter", "fixed"] /2, # battery inverter represented by two links (charging and discharging), while costs in cost data are for bidirectional inverter --> correction here
    p_nom_extendable=True,
    lifetime=costs.at["battery inverter", "lifetime"],
    p_min_pu=0,
    p_max_pu=1,
)

n.madd(
    "Link",
    nodes + " battery discharger",
    bus0=nodes + " battery",
    bus1=nodes,
    carrier="battery discharger",
    efficiency=costs.at["battery inverter", "efficiency"] ** 0.5,
    marginal_cost=yaml_content["sector"]["marginal_cost_storage"] /2, # battery inverter represented by two links (charging and discharging), while costs in cost data are for bidirectional inverter --> correction here
    p_nom_extendable=True,
    lifetime=costs.at["battery inverter", "lifetime"],
    p_min_pu=0,
    p_max_pu=1,
)

Index(['TR.41_1_AC battery discharger', 'TR.44_1_AC battery discharger',
       'TR.58_1_AC battery discharger', 'TR.63_1_AC battery discharger',
       'TR.68_1_AC battery discharger', 'TR.8_1_AC battery discharger'],
      dtype='object', name='Bus')

### Add H2 pipelines

In [19]:
# List of all AC nodes for export ports
nodes_df_export = n.buses[(n.buses.carrier == "AC") & (n.buses.location.isin(exp_ports.index))]
nodes_export = nodes_df_export.index

nodes_df = n.buses[n.buses.carrier == "AC"]

# Convert to GeoDataFrame
nodes_geodf_export = gpd.GeoDataFrame(
    nodes_df_export, 
    geometry=gpd.points_from_xy(nodes_df_export['x'], nodes_df_export['y']),  # Create geometry column
    crs="EPSG:4326"  # Set the coordinate reference system (WGS84 in this case)
)
nodes_geodf_export.index.name = 'Bus1'

# Convert CRS to EPSG:3857 so we can measure distances
nodes_geodf_export = nodes_geodf_export.to_crs(epsg=3857)


# Remove nodes in nodes_df that exist in nodes_df_export
nodes_df_filtered = nodes_df.loc[nodes_df.index.difference(nodes_df_export.index)]

if not nodes_df_filtered.empty:
    # Proceed with the filtered nodes
    nodes_geodf = gpd.GeoDataFrame(
        nodes_df_filtered, 
        geometry=gpd.points_from_xy(nodes_df_filtered['x'], nodes_df_filtered['y']),  # Create geometry column
        crs="EPSG:4326"  # Set the coordinate reference system (WGS84 in this case)
    )

    # nodes_geodf = gpd.GeoDataFrame(
    #     nodes_df, 
    #     geometry=gpd.points_from_xy(nodes_df['x'], nodes_df['y']),  # Create geometry column
    #     crs="EPSG:4326"  # Set the coordinate reference system (WGS84 in this case)
    # )
    nodes_geodf.index.name = 'Bus0'

    # Convert CRS to EPSG:3857 so we can measure distances
    nodes_geodf = nodes_geodf.to_crs(epsg=3857)

    # Compute distances between all points in gdf1 and gdf2
    distances = nodes_geodf.geometry.apply(lambda p1: nodes_geodf_export.geometry.distance(p1))

    # Convert the result into a DataFrame for better readability
    h2_links = distances.apply(pd.Series)

    # Convert the distance DataFrame to long format
    h2_links = h2_links.stack().reset_index()

    # Rename columns
    h2_links.columns = ['bus0', 'bus1', 'length']

    # Convert length to Km
    h2_links['length'] = h2_links['length'] /1e3

    # Set the index to a formatted string representing the pipeline connection
    h2_links.index = h2_links.apply(lambda row: f"H2 pipeline {row['bus0']} -> {row['bus1']}", axis=1)



    n.madd(
        "Link",
        h2_links.index,
        bus0=h2_links.bus0.values + " H2",
        bus1=h2_links.bus1.values + " H2",
        p_min_pu=-1,
        # p_nom_min=snakemake.params.p_nom_min_pipeline,
        p_nom_extendable=True,
        length=h2_links.length.values,
        capital_cost=costs.at["H2 (g) pipeline", "fixed"] * h2_links.length.values,
        carrier="H2 pipeline",
        lifetime=costs.at["H2 (g) pipeline", "lifetime"],
        terrain_factor=yaml_content["costs"]["pipelines"]["length_factor"],
        # efficiency=costs.at["H2 (g) pipeline", "efficiency"] ** h2_links.length.values,
    )

### Attach ESC

In [20]:
additional_costs_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/additional_costs_2050.csv"
additional_costs_input = prepare_costs(
    additional_costs_path,
    yaml_content["costs"]["USD2013_to_EUR2013"],
    dr,
    Nyears,
    yaml_content["costs"]["lifetime"],
)

costs = pd.concat([costs, additional_costs_input], ignore_index=False)



# export_ports_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/export_ports.csv"
# ports = pd.read_csv(
#     export_ports_path,
#     index_col=None,
#     keep_default_na=False,
# ).squeeze()

# ports = ports[ports.country.isin(country)]

# exp_ports = ports.copy()
# exp_ports.set_index("name", inplace=True)
# exp_ports["bus"] = 0

# # Add 3 export locations to attach ESC to them
# n.madd(
#     "Bus",
#     exp_ports.index + " TR_AC",
#     carrier="port_AC",
#     x=exp_ports.x.values,
#     y=exp_ports.y.values,
#     location=exp_ports.index + " TR_AC",
#     country='TR',
# )

# n.madd(
#     "Link",
#     exp_ports.index + " TR_AC" + " port",
#     bus1=exp_ports.index + " TR_AC",
#     bus0=nodes,
#     p_nom_extendable=True,
#     carrier="AC",
# )
# n.madd(
#     "Bus",
#     exp_ports.index + " TR_AC H2",
#     carrier="port_H2",
#     x=exp_ports.x.values,
#     y=exp_ports.y.values,
#     location=exp_ports.index + " TR_AC",
#     country='TR',
# )

# n.madd(
#     "Link",
#     exp_ports.index + " TR_AC H2" + " port",
#     bus1=exp_ports.index + " TR_AC H2",
#     bus0=nodes + " H2",
#     p_nom_extendable=True,
#     carrier="H2",
# )

# exp_ports.loc[exp_ports.index,"bus"]=exp_ports.index + " TR_AC"
# exp_ports.set_index("bus", inplace=True)

# # get hydrogen export buses/ports
# hydrogen_buses_ports = n.buses[n.buses.carrier == "port_H2"]

# # List of ports electrictiy buses
# exp_nodes = hydrogen_buses_ports.location.values

# # List of all AC nodes
# nodes_df = n.buses[n.buses.carrier == "AC"]
# nodes = n.buses[n.buses.carrier == "AC"].index

import_ports_path = "/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/import_ports.csv"

# Import ports and nodes
import_ports = pd.read_csv(
    import_ports_path,
    index_col=None,
    keep_default_na=False,
)#.squeeze()

import_ports.set_index('name', inplace=True)
imp_nodes = import_ports.index

# Get the shipping Distances between the import and export ports
# get_shipping_distances(exp_ports, import_ports)


In [21]:

# Export Supply Chain wildcard
export_esc = yaml_content["export"]["esc_scenarios"]["esc"][0]  

In [22]:
# Create import profile
import_profiles = create_import_profile(sopts, h2export, yaml_content)

INFO:helpers_offgrid:The yearly import demand is 150.0 TWh, profile generated based on esc_scenarios method and resampled to 3H


In [23]:
create_esc_network(n, nodes_export, exp_nodes, imp_nodes, exp_ports, export_esc, import_profiles, import_ports, nodes_df_export, yaml_content, efficiencies, costs)

/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/scripts/helpers_offgrid.py:1478: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/scripts/helpers_offgrid.py:1483: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  import_country = world[world['iso_2'] == import_ports.country[0]]
/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/scripts/helpers_offgrid.py:1490: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a

In [24]:
add_esc_shipping(n, dr, efficiencies_path, yaml_content, export_esc, costs_path)

INFO:helpers_offgrid:Increasing the round-trip travel time from 428h to 486h (+13.55%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 10 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 426h to 486h (+14.08%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 10 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 422h to 486h (+15.17%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 10 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 464h to 486h (+4.74%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 10 shipping convoys to shipping route.
INFO:helpers_offgrid:Increasing the round-trip travel time from 460h to 486h (+5.65%) to achieve more levelled supply by ship.
INFO:helpers_offgrid:Adding 10 shipping convoys to shipping route.
INFO:helpers_offgrid:Increa

PyPSA Network 'PyPSA-Earth'
Components:
 - Bus: 232
 - Carrier: 11
 - Generator: 18
 - Link: 348
 - Load: 1
 - Store: 114
Snapshots: 2920

In [25]:
n

PyPSA Network 'PyPSA-Earth'
Components:
 - Bus: 232
 - Carrier: 11
 - Generator: 18
 - Link: 348
 - Load: 1
 - Store: 114
Snapshots: 2920

In [26]:
n.links[n.links.carrier=='H2 pipeline']

,bus1,bus0,p_nom_extendable,carrier,efficiency,capital_cost,lifetime,build_year,bus2,bus3,...,ramp_limit_down,ramp_limit_shut_down,ramp_limit_start_up,ramp_limit_up,shut_down_cost,start_up_cost,terrain_factor,type,up_time_before,scale_costs_based_on
Link,,,,,,,,,,,,,,,,,,,,,
H2 pipeline TR.44_1_AC -> TR.41_1_AC,TR.41_1_AC H2,TR.44_1_AC H2,True,H2 pipeline,1.0,22530.990957,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.44_1_AC -> TR.58_1_AC,TR.58_1_AC H2,TR.44_1_AC H2,True,H2 pipeline,1.0,3532.246615,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.44_1_AC -> TR.63_1_AC,TR.63_1_AC H2,TR.44_1_AC H2,True,H2 pipeline,1.0,22226.949024,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.68_1_AC -> TR.41_1_AC,TR.41_1_AC H2,TR.68_1_AC H2,True,H2 pipeline,1.0,43113.810935,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.68_1_AC -> TR.58_1_AC,TR.58_1_AC H2,TR.68_1_AC H2,True,H2 pipeline,1.0,18368.693334,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.68_1_AC -> TR.63_1_AC,TR.63_1_AC H2,TR.68_1_AC H2,True,H2 pipeline,1.0,21635.152606,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.8_1_AC -> TR.41_1_AC,TR.41_1_AC H2,TR.8_1_AC H2,True,H2 pipeline,1.0,15651.579436,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.8_1_AC -> TR.58_1_AC,TR.58_1_AC H2,TR.8_1_AC H2,True,H2 pipeline,1.0,10768.974331,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN
H2 pipeline TR.8_1_AC -> TR.63_1_AC,TR.63_1_AC H2,TR.8_1_AC H2,True,H2 pipeline,1.0,27930.334144,50.0,0.0,,,...,NaN,1.0,1.0,NaN,0.0,0.0,1.2,,1,NaN


In [27]:
n.links[n.links.carrier=='NH3 pipeline']

,bus1,bus0,p_nom_extendable,carrier,efficiency,capital_cost,lifetime,build_year,bus2,bus3,...,ramp_limit_down,ramp_limit_shut_down,ramp_limit_start_up,ramp_limit_up,shut_down_cost,start_up_cost,terrain_factor,type,up_time_before,scale_costs_based_on
Link,,,,,,,,,,,,,,,,,,,,,


In [28]:
yaml_content["export"]["esc_scenarios"]["synthesis"]

'at_port'

In [29]:
n.lines

attribute,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,v_ang_min,v_ang_max,sub_network,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt
Line,,,,,,,,,,,,,,,,,,,,,


In [30]:
n.links[n.links.carrier.str.contains('Haber-Bosch')] 

,bus1,bus0,p_nom_extendable,carrier,efficiency,capital_cost,lifetime,build_year,bus2,bus3,...,ramp_limit_down,ramp_limit_shut_down,ramp_limit_start_up,ramp_limit_up,shut_down_cost,start_up_cost,terrain_factor,type,up_time_before,scale_costs_based_on
Link,,,,,,,,,,,,,,,,,,,,,
TR.41_1_AC Haber-Bosch,TR.41_1_AC NH3 (g),TR.41_1_AC,True,NH3 Haber-Bosch,4.043672,117366.941033,30.0,0.0,TR.41_1_AC N2 (g),TR.41_1_AC H2,...,NaN,1.0,1.0,NaN,0.0,0.0,1.0,,1,bus1
TR.58_1_AC Haber-Bosch,TR.58_1_AC NH3 (g),TR.58_1_AC,True,NH3 Haber-Bosch,4.043672,117366.941033,30.0,0.0,TR.58_1_AC N2 (g),TR.58_1_AC H2,...,NaN,1.0,1.0,NaN,0.0,0.0,1.0,,1,bus1
TR.63_1_AC Haber-Bosch,TR.63_1_AC NH3 (g),TR.63_1_AC,True,NH3 Haber-Bosch,4.043672,117366.941033,30.0,0.0,TR.63_1_AC N2 (g),TR.63_1_AC H2,...,NaN,1.0,1.0,NaN,0.0,0.0,1.0,,1,bus1


### Solve

In [31]:
from helpers_offgrid import (
    prepare_network,
    solve_network,
)

from solve_network import (
    add_battery_constraints,
    add_nh3_store_cap,
)

from pypsa.linopf import ilopf, network_lopf
from pypsa.linopt import define_constraints, get_var, join_exprs, linexpr

from vresutils.benchmark import memory_logger

In [32]:
# tmpdir = yaml_content["solving"].get("tmpdir")

# if tmpdir is not None:
#     Path(tmpdir).mkdir(parents=True, exist_ok=True)
#     opts = yaml_content["scenario"]["opts"][0].split("-")
#     solve_opts = yaml_content["solving"]["options"]

# # fn ="/nimble/home/edd32710/projects/kikikiki/memory.log"
# # with memory_logger(filename=fn, interval=30.0) as mem:
# #     overrides = override_component_attrs("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/override_component_attrs")
# #     n = pypsa.Network(n, override_component_attrs=overrides)



#     n = prepare_network(n, solve_opts)

#     n = solve_network(
#         n,
#         config=yaml_content,
#         yaml_content=yaml_content,
#         opts=yaml_content["scenario"]["opts"][0].split("-"),
        
#         # solver_dir=tmpdir,
#         # solver_logfile=snakemake.log.solver,
#     )
    
#     n.meta = dict(yaml_content)
#     n.export_to_netcdf("/nimble/home/edd32710/projects/kikikiki/trial1.nc")

# # # logging output to the terminal
# # print("Objective function: {}".format(n.objective))

# # # logging output to file
# # logger.info("Objective function: {}".format(n.objective))
# # logger.info("Objective constant: {}".format(n.objective_constant))
# # logger.info("Maximum memory usage: {}".format(mem.mem_usage))

In [33]:
# Path(tmpdir).mkdir(parents=True, exist_ok=True)
opts = yaml_content["scenario"]["opts"][0].split("-")
solve_opts = yaml_content["solving"]["options"]

# fn ="/nimble/home/edd32710/projects/kikikiki/memory.log"
# with memory_logger(filename=fn, interval=30.0) as mem:
#     overrides = override_component_attrs("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/data/override_component_attrs")
#     n = pypsa.Network(n, override_component_attrs=overrides)



n = prepare_network(n, solve_opts)

n = solve_network(
    n,
    config=yaml_content,
    yaml_content=yaml_content,
    opts=yaml_content["scenario"]["opts"][0].split("-"),
    
    # solver_dir=tmpdir,
    # solver_logfile=snakemake.log.solver,
)

n.meta = dict(yaml_content)
# n.export_to_netcdf("/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_off_grid_{}_20240927/postnetworks/elec_s_348_ec_lv1.1_Co2L_3H_2050_0.091_AP_{}export_shipping_lnh3.nc".format(offgrid_scenario, h2export))


n.export_to_netcdf("/nimble/home/edd32710/projects/offgrid_islanded_6nodes/elec_s_480_ec_lv1.1_Co2L_3H_2050_0.091_AP_{}export_shipping_lnh3.nc".format(h2export))


INFO:pypsa.linopf:Prepare linear problem
INFO:pypsa.linopf:Total preparation time: 19.2s
INFO:pypsa.linopf:Solve linear problem using Gurobi solver


Set parameter TokenServer to value "10.186.19.42"
Read LP format model from file /tmp/pypsa-problem-j3u1zvav.lp
Reading time = 7.52 seconds
obj: 3813527 rows, 1734961 columns, 7248864 nonzeros
Set parameter Threads to value 25
Set parameter Method to value 2
Set parameter Crossover to value 0
Set parameter BarConvTol to value 1e-06
Set parameter Seed to value 123
Set parameter AggFill to value 0
Set parameter PreDual to value 0
Set parameter GURO_PAR_BARDENSETHRESH to value 200
Variables with '<=' in their names or with non-numeric values:

Constraints with '<=' in their names or with non-numeric RHS values:
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: AMD EPYC 7542 32-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 32 physical cores, 64 logical processors, using up to 25 threads

Optimize a model with 3813527 rows, 1734961 columns and 7248864 nonzeros
Model fingerprint: 0x55a7da58
Coefficient statistics:
  Matrix range    

INFO:pypsa.io:Exported network elec_s_480_ec_lv1.1_Co2L_3H_2050_0.091_AP_150export_shipping_lnh3.nc has links, loads, buses, stores, generators, carriers


<xarray.Dataset>
Dimensions:                       (snapshots: 2920, investment_periods: 0,
                                   links_i: 348, links_t_p_max_pu_i: 180,
                                   links_t_p_min_pu_i: 180, loads_i: 1,
                                   loads_t_p_set_i: 1, buses_i: 232,
                                   stores_i: 114, generators_i: 18,
                                   generators_t_p_max_pu_i: 18, carriers_i: 11)
Coordinates:
  * snapshots                     (snapshots) int64 0 1 2 3 ... 2917 2918 2919
  * investment_periods            (investment_periods) int64 
  * links_i                       (links_i) object 'TR.41_1_AC H2 Electrolysi...
  * links_t_p_max_pu_i            (links_t_p_max_pu_i) object 'NH3 (l) transp...
  * links_t_p_min_pu_i            (links_t_p_min_pu_i) object 'NH3 (l) transp...
  * loads_i                       (loads_i) object 'NH3 export load'
  * loads_t_p_set_i               (loads_t_p_set_i) object 'NH3 export load'
  * buses_i                       (buses_i) object 'TR.41_1_AC' ... 'NH3 (l) ...
  * stores_i                      (stores_i) object 'TR.41_1_AC H2 UHS' ... '...
  * generators_i                  (generators_i) object 'TR.41_1_AC solar' .....
  * generators_t_p_max_pu_i       (generators_t_p_max_pu_i) object 'TR.41_1_A...
  * carriers_i                    (carriers_i) object 'H2' 'OCGT' ... 'N2'
Data variables: (12/55)
    snapshots_snapshot            (snapshots) datetime64[ns] 2013-01-01 ... 2...
    snapshots_objective           (snapshots) float64 3.0 3.0 3.0 ... 3.0 3.0
    snapshots_stores              (snapshots) float64 3.0 3.0 3.0 ... 3.0 3.0
    snapshots_generators          (snapshots) float64 3.0 3.0 3.0 ... 3.0 3.0
    investment_periods_objective  (investment_periods) object 
    investment_periods_years      (investment_periods) object 
    ...                            ...
    generators_lifetime           (generators_i) float64 20.0 20.0 ... 20.0 20.0
    generators_control            (generators_i) object 'Slack' 'Slack' ... 'PQ'
    generators_marginal_cost      (generators_i) float64 0.00992 ... 0.0091
    generators_t_p_max_pu         (snapshots, generators_t_p_max_pu_i) float64 ...
    carriers_color                (carriers_i) object '#ea048a' '#d35050' ... ''
    carriers_nice_name            (carriers_i) object 'Hydrogen Storage' ... ...
Attributes:
    network__cCounter:           3813528
    network__multi_invest:       0
    network__xCounter:           1734962
    network_name:                PyPSA-Earth
    network_objective:           nan
    network_objective_constant:  0.0
    network_pypsa_version:       0.24.0
    network_srid:                4326
    meta:                        {"logging_level": "INFO", "tutorial": false,...

In [34]:
/nimble/home/edd32710/projects/Paper_1/pypsa-earth-sec/results/TR_2050_shipping_lnh3_off_gridRESprofileatport_20240927/postnetworks

NameError: name 'nimble' is not defined

In [ ]:
yaml_content["export"]["esc_scenarios"]["synthesis"] == 'free'

False

In [ ]:
'NH3 export load' in n.loads_t['p'].columns

True

In [ ]:
n.links_t['p3'].filter(like='Haber-Bosch')

Link,TR.63_1_AC Haber-Bosch,TR.58_1_AC Haber-Bosch,TR.41_1_AC Haber-Bosch
snapshot,,,
2013-01-01 00:00:00,1.122412e-08,543.090133,7.682363e-08
2013-01-01 03:00:00,1.131966e-08,543.090133,7.649620e-08
2013-01-01 06:00:00,1.635842e-08,1663.030407,8.451085e-08
2013-01-01 09:00:00,1.628376e-08,1701.543404,9.288188e-08
2013-01-01 12:00:00,1.645748e-08,1678.809607,9.158482e-08
...,...,...,...
2013-12-31 09:00:00,1.099538e-08,1687.940971,9.317681e-08
2013-12-31 12:00:00,1.100702e-08,1650.364655,9.230707e-08
2013-12-31 15:00:00,1.102178e-08,543.090133,7.713586e-08


In [ ]:
yaml_content["sector"]["ammonia"]

{'network_limit': 48612, 'storage_limit': 0}

In [ ]:
n.stores.loc[(n.stores.carrier == "NH3 store")].e_nom_opt.sum()/1e6

1.157519240440167

In [ ]:
n.generators.p_nom_opt

Generator
TR.63_1_AC solar      4.192040e-07
TR.58_1_AC solar      1.333919e+04
TR.41_1_AC solar      1.067758e-06
TR.63_1_AC onwind     1.001133e-08
TR.58_1_AC onwind     8.784574e-09
TR.41_1_AC onwind     2.669775e-08
TR.63_1_AC onwind2    3.113695e-09
TR.58_1_AC onwind2    1.452079e-09
TR.41_1_AC onwind2    3.843506e-09
Name: p_nom_opt, dtype: float64

In [ ]:
n.generators.p_nom_max

Generator
TR.63_1_AC solar      28181.6710
TR.58_1_AC solar      43899.6000
TR.41_1_AC solar      26577.8900
TR.63_1_AC onwind     10581.0000
TR.58_1_AC onwind     14505.9000
TR.41_1_AC onwind     13977.5000
TR.63_1_AC onwind2       44.4388
TR.58_1_AC onwind2      175.1170
TR.41_1_AC onwind2      152.5540
Name: p_nom_max, dtype: float64

In [ ]:
n.generators_t.p_max_pu#.filter(like='solar', axis=1)

Generator,TR.41_1_AC onwind,TR.41_1_AC onwind2,TR.41_1_AC solar,TR.58_1_AC onwind,TR.58_1_AC onwind2,TR.58_1_AC solar,TR.63_1_AC onwind,TR.63_1_AC onwind2,TR.63_1_AC solar
snapshot,,,,,,,,,
2013-01-01 00:00:00,0.471632,0.000000,0.000000,0.000000,0.0,0.000000,0.391249,0.048902,0.000000
2013-01-01 03:00:00,0.548139,0.023256,0.000000,0.000000,0.0,0.000000,0.414304,0.067170,0.000000
2013-01-01 06:00:00,0.648255,0.043431,0.158019,0.014881,0.0,0.242858,0.441045,0.076304,0.252966
2013-01-01 09:00:00,0.580716,0.033048,0.529820,0.025997,0.0,0.428665,0.404960,0.046618,0.499680
2013-01-01 12:00:00,0.725213,0.079223,0.391441,0.066237,0.0,0.255043,0.398075,0.083155,0.171329
...,...,...,...,...,...,...,...,...,...
2013-12-31 09:00:00,0.070303,0.000000,0.544192,0.000000,0.0,0.380287,0.017219,0.000000,0.048224
2013-12-31 12:00:00,0.108348,0.000000,0.435245,0.000000,0.0,0.244209,0.017722,0.000000,0.030136
2013-12-31 15:00:00,0.106353,0.000000,0.024366,0.000000,0.0,0.000000,0.011335,0.000000,0.000000


In [ ]:
n.links

,bus1,bus0,p_nom_extendable,carrier,efficiency,capital_cost,lifetime,build_year,bus2,bus3,...,ramp_limit_shut_down,ramp_limit_start_up,ramp_limit_up,shut_down_cost,start_up_cost,terrain_factor,type,up_time_before,scale_costs_based_on,charger_ratio
Link,,,,,,,,,,,,,,,,,,,,,
TR_AC H2 Electrolysis,TR_AC H2,TR_AC,True,H2 Electrolysis,0.750000,26159.892648,35.0,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC H2 Fuel Cell,TR_AC,TR_AC H2,True,H2 Fuel Cell,0.500000,82602.649562,10.0,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC H2 UHS charger,TR_AC H2 UHS,TR_AC H2,True,H2 UHS charger,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC H2 UHS discharger,TR_AC H2,TR_AC H2 UHS,True,H2 UHS discharger,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
TR_AC battery charger,TR_AC battery,TR_AC,True,battery charger,0.979796,4965.198717,10.0,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,-2482.633938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NH3 (l) transport ship convoy 2 - Aliaga TR_AC --> Damietta unloading,Damietta berth (imp),NH3 (l) transport ship convoy 2 - Aliaga TR_AC...,True,NH3,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
NH3 (l) transport ship convoy 2 - Aliaga TR_AC --> Damietta trip demand & losses,NH3 (l) transport ship convoy 2 - Aliaga TR_AC...,NH3 (l) transport ship convoy 2 - Aliaga TR_AC...,True,NH3,0.995921,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN
NH3 (l) transport ship convoy 3 - Aliaga TR_AC --> Damietta loading,NH3 (l) transport ship convoy 3 - Aliaga TR_AC...,Aliaga TR_AC berth (exp),True,NH3,1.000000,0.000000,inf,0.0,,,...,1.0,1.0,NaN,0.0,0.0,1.0,,1,NaN,NaN


In [ ]:
n.objective

870314218.9234629

In [ ]:
n

PyPSA Network 'PyPSA-Earth'
Components:
 - Bus: 90
 - Carrier: 11
 - Generator: 9
 - Link: 123
 - Load: 1
 - Store: 40
 - SubNetwork: 90
Snapshots: 2920

In [ ]:
from pes_analysis_helpers import*

In [ ]:
calc_ptx_demand(n)

0.0

In [ ]:
calc_anh3s_capa_exp(n, agg=True)

8.951430158372843e-17

In [ ]:
calc_batt_capa(n)

6.110470001330325e-21

In [ ]:
calc_uhs_capa(n, agg=True)

5.531519561497319e-19

In [ ]:
'NH3 export load' in n.loads_t['p'].columns

True

In [ ]:
if 'NH3 export load' in n.loads_t['p'].columns:
    d_h2 = n.links_t['p3'].filter(like='Haber-Bosch') #Amounts of H2 produced at all nodes
    d_h2.columns = n.links.loc[d_h2.columns, 'bus3']
elif 'H2 export load' in n.loads_t['p'].columns:
    d_h2 = n.links_t['p0'].filter(like='H2 liquefaction') #Amounts of H2 produced at all nodes
    d_h2.columns = n.links.loc[d_h2.columns, 'bus0']
elif 'meOH export load' in n.loads_t['p'].columns:
    d_h2 = n.links_t['p1'].filter(like='methanolisation') #Amounts of H2 produced at all nodes
    d_h2.columns = n.links.loc[d_h2.columns, 'bus1']      

conversion_fact=33.3 # MWh/t_H2

weightings = pd.DataFrame(
    np.outer(n.snapshot_weightings["generators"], [1.0] * len(d_h2.T)),
    index=n.snapshots,
    columns=d_h2.columns,
)
d_h2 = d_h2 * weightings
# d_h2.columns = n.links.loc[d_h2.columns, 'bus0']
d_h2 = d_h2.groupby(d_h2.columns, axis=1).sum()


marginal_prices = n.buses_t.marginal_price.loc[:, d_h2.columns] #Marginal H2 prices at export nodes
h2_costs = (d_h2 * marginal_prices).sum()

/tmp/ipykernel_3702965/2232106857.py:20: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  d_h2 = d_h2.groupby(d_h2.columns, axis=1).sum()


In [ ]:
d_h2

bus3,Aliaga TR_AC H2,Mersin TR_AC H2,Samsun TR_AC H2
snapshot,,,
2013-01-01 00:00:00,5.967634e-21,5.872129e-21,6.266596e-21
2013-01-01 03:00:00,5.966458e-21,5.870601e-21,6.275705e-21
2013-01-01 06:00:00,5.970974e-21,5.879133e-21,6.280246e-21
2013-01-01 09:00:00,5.966824e-21,5.876554e-21,6.276960e-21
2013-01-01 12:00:00,5.966088e-21,5.872823e-21,6.274867e-21
...,...,...,...
2013-12-31 09:00:00,5.953029e-21,5.861861e-21,6.257920e-21
2013-12-31 12:00:00,5.966632e-21,5.874049e-21,6.278078e-21
2013-12-31 15:00:00,5.974846e-21,5.877424e-21,6.289721e-21
